In [1]:
import pickle
import os
from IPython.core.display import display, HTML
import pandas as pd
import sys
import re
from collections import Counter
import shutil

In [2]:
table_dir = '../data/'
string_matched_data = pickle.load(open(os.path.join(table_dir, 'all_string_match_data_pre.pkl'), 'rb'))
# string_matched_data = []
## skip to ds block

In [3]:
print(len(string_matched_data))
print(type(string_matched_data))
string_matched_data[4]

596
<class 'list'>


{'act_table': [['x mol%',
   'Tg1  +-3',
   'Tg2  (degC) +-3',
   'Tx (degC) +-3',
   'Tp (degC) +-3',
   'DT (degC) +-6'],
  ['27.5', '750', '810', '954', '971', '204'],
  ['30', '759', '805', '973', '987', '214'],
  ['32.5', '761', '811', '1011', '1036', '250'],
  ['35', '777', '807', '980,1010', '992,1037', '203'],
  ['37.5', '797', '812', '1007', '1031', '210']],
 'caption': 'The characteristic temperatures (Tg, Tx and Tp) and DT criteria for Gd2O3-SiO2-B2O3 glasses with various Gd2O3 concentrations.',
 'row_label': [0, 0, 0, 0, 0, 1],
 'col_label': [2, 0, 0, 13, 13, 0],
 'edge_list': [(30, 0)],
 'pii': 'S0022309314001069',
 't_idx': 1,
 'regex_table': 0,
 'num_rows': 6,
 'num_cols': 6,
 'num_cells': 36,
 'comp_table': True,
 'input_ids': [[102, 412, 3903, 1863, 103],
  [102, 7720, 30130, 473, 579, 239, 103],
  [102, 7720, 30132, 145, 1859, 30116, 546, 473, 579, 239, 103],
  [102, 8951, 145, 1859, 30116, 546, 473, 579, 239, 103],
  [102, 6972, 145, 1859, 30116, 546, 473, 579, 239, 

In [4]:
# check_table = [['Glass',
#    'Tg (degC)',
#    'Thermal conductivity (Wm-1 k-1)',
#    'Thermal expansion coefficient (x10-7 degC-1)',
#    "Young's modulus (GPa)"],
#   ['Phosphate glass (K-PSK200, Sumita Optical Glass, Inc.)',
#    '387',
#    '0.7',
#    '123',
#    '72'],
#   ['Borosilicate glass (L-BAL42, Ohara Inc.)', '0.0', '1.0', '0', '89']]
# check = get_nums(check_table)
# check

In [5]:
def find_num(string):
    #remove tabs or unnecessary spaces
    try:
        string = string.strip()
    except:
        pass
    #e.g. already in int or float form: 12.5 -> 12.5
    try:
        if string[0] == '-':
            return float(string[1:])*(-1)
        else:
            return float(string)
    except:
        pass
    # e.g. 99.1x10-7
    range_regex = re.compile('^\d+\.?\d*\s*x\s*\d+\.?\d*[-|+]?\s*\d+\.?\d*$')
    try:
#         print('alla')
        ranges = range_regex.search(string).group().split('x')
        if '-' in ranges[1]:
            numu = ranges[1].split('-')
#             print(ranges)
#             print(float(numu[0]), float(numu[1]))
            num = float(ranges[0]) * pow(float(numu[0]), (-float(numu[1])))
            formatted_result = "{:.3e}".format(num)
            return formatted_result
        elif '+' in ranges[1]:
            numu = ranges[1].split('+')
            num = float(ranges[0]) * pow(float(numu[0]), (float(numu[1])))
#             print(num)
            return num
        num = float(ranges[0]) * float(ranges[1])
        formatted_result = "{:.3e}".format(num)
        return formatted_result
    except:
        pass
    #e.g. 12.5 - 13.5 -> 12.5
    range_regex = re.compile('^\d+\.?\d*\s*-\s*\d+\.?\d*')
    try:
        ranges = range_regex.search(string).group().split('-')
        num = float(ranges[0])
#         print(num)
        return num
    except:
        pass
#     print('alla')
    #e.g. 12.2 (5.2) -> 12.2
    bracket_regex = re.compile('(\d+\.?\d*)\s*\(\d*.?\d*\)')
    try:
        extracted_value = float(bracket_regex.search(string).group(1))
        return float(extracted_value)
    except:
        pass
    #e.g. 12.3 ± 0.5 -> 12.3
    plusmin_regex = re.compile('^(\d+\.?\d*)(\s*[±+-]+\s*\d+\.?\d*)')
    try:
        extracted_value = float(plusmin_regex.search(string).group(1))
        return extracted_value
    except AttributeError:
        pass
    #e.g. <0.05 -> 0.05  |  >72.0 -> 72.0    | ~12 -> 12
    lessthan_roughly_regex = re.compile('([<]|[~]|[>])=?\s*\d+\.*\d*')
    try:
        extracted_value = lessthan_roughly_regex.search(string).group()
        num_regex = re.compile('\d+\.*\d*')
        extracted_value = num_regex.search(extracted_value).group()
        return float(extracted_value)
    except:
        pass
    # e.g. 0.4:0.6 (ratios)
    if ':' in string:
        split = string.split(":")
        try:
            extracted_value = round(float(split[0])/float(split[1]), 3)
            return extracted_value
        except:
            pass
    # e.g. 220 [29] --> 220.0, where citations given, rejecting ab 220 [29] or 7 220 [29]
    if '[' in string and ']' in string:
        try:
            extracted_value = string[:string.index('[')]
            return float(extracted_value)
        except:
            pass
    # e.g. 723 (first peak or other text) --> 723.0
    if '(' in string and ')' in string:
        try:
            extracted_value = string[:string.index('(')]
            return float(extracted_value)
        except:
            pass
    # e.g. '150K' or '150 degC' --> 150.0 but not '1350degC/2h' or 'njn 150 njn'
    try:
        exact_number_regex = re.compile('^(\d+\.?\d*)\s*[a-zA-Z]+$')
        # Using search to find the first occurrence of the pattern in the string
        match = exact_number_regex.search(string)
#         print('Match')
#         print(match)
        
        # If a match is found, extract the numeric part and convert to float
        if match:
            numeric_part = match.group(1)
            extracted_value = float(numeric_part)
            return extracted_value
    except:
        pass
    return None

In [6]:
num_pattern = re.compile(r'\d*\.\d+|\d+')

def num_post_process(num):
    return float(num)

# old get_nums
# def get_nums(table):
#     nums = []
#     for r in table:
#         r_comps, r_nums = [], []
#         for cell in r:
#             # cell_nums = re.findall(num_pattern, re.sub(cons_pattern, ' ', cell))
#             cell_nums = re.findall(num_pattern, cell)
#             r_nums.append(list(set(map(num_post_process, cell_nums))))
#         nums.append(r_nums)
#     # print(f'nums {nums}')
#     return nums

# new get_nums:
def get_nums(table):
    nums = []
    for r in table:
        r_nums = []
        for cell in r:
            num = find_num(cell)
            if num != None and num != 0:
                cell_nums = [find_num(cell)]
            else:
                cell_nums = []
            r_nums.append(list(set(map(num_post_process, cell_nums))))
        nums.append(r_nums)
    # print(f'nums {nums}')
    return nums

In [7]:
prop_ids = [510, 1140, 2010, 2051, 540, 50, 180, 60, 160, 1014, 1015, 3012, 3174, 1116, 1119, 1020, 1011, 1118, 70, 1306]
prop_names = ['Density', 'GlassTransitionTg', 'RefractiveIndex', 'AbbeValue', 'YoungsModulus', 'ShearModulus', 'VickersHardness', 'PoissonRatio', 'FractureToughness', 'CrystallizationTemp', 'MeltingTemp', 'ElectricConduct', 'DielectricConst', 'TSofP', 'TAnnealingP', 'ExpansionCoeff', 'LiquidusTemperature', 'SofP', 'BulkModulus', 'ActivationEnergy']

In [8]:
def get_dimensions(lst, current_dimensions=None):
    if current_dimensions is None:
        current_dimensions = []

    if not isinstance(lst, list):
        return current_dimensions

    current_dimensions.append(len(lst))

    if len(lst) == 0 or not any(isinstance(item, list) for item in lst):
        return current_dimensions

    return get_dimensions(lst[0], current_dimensions)

In [9]:
for index, table in enumerate(string_matched_data):
    table['prop_table'] = False
    table['prop_names'] = set()
    table['prop_orient'] = ''
    table['props'] = []
    table['prop_row_col_indexes'] = []
    table['table_nums'] = get_nums(table['act_table'])
    table['anno_type'] = 'string_match'
    
    row_length, col_length = table['num_rows'], table['num_cols']
    row_label, col_label = table['row_label'], table['col_label']
    row_col_label = row_label + col_label
        
    if any(elem>=4 for elem in col_label) and any(elem>=4 for elem in row_label):
        print(table)
        System.exit('There is some error, cannot be both orientation')
    if any(elem>=4 for elem in col_label):
        table['prop_orient'] = 'col'
    elif any(elem>=4 for elem in row_label):
        table['prop_orient'] = 'row'
    else:
        System.exit('There is some error')
        
    if any(elem>=4 for elem in row_col_label):
        table['prop_table'] = True
    else:
        System.exit('There is some error')
        
            
    for i in range(4, 24):
        table['props'].append([[[] for j in range(table['num_cols'])] for i in range(table['num_rows'])])
        table['prop_row_col_indexes'].append([])
        temp_list = []
        found = ''

        # Check if i is in col_label
        if i in col_label and table['prop_orient'] == 'col':
            table['prop_names'].add(prop_names[i-4])
            col_indexes = [index for index, value in enumerate(col_label) if value == i]
            for col_index in col_indexes:
                table['prop_row_col_indexes'][-1].append(col_index)
                for r in range(1, row_length):
                    if len(table['table_nums'][r][col_index])>0:
                        table['props'][-1][r][col_index].append((prop_ids[i-4], table['table_nums'][r][col_index][0], ''))

        # Check if i is in row_label
        elif i in row_label and table['prop_orient'] == 'row':
            table['prop_names'].add(prop_names[i-4])
            row_indexes = [index for index, value in enumerate(row_label) if value == i]
            for row_index in row_indexes:
                table['prop_row_col_indexes'][-1].append(row_index)
                for c in range(1, col_length):
                    if len(table['table_nums'][row_index][c])>0:
                        table['props'][-1][row_index][c].append((prop_ids[i-4], table['table_nums'][row_index][c][0], ''))
                    
#     if index<5:
#         dimensions = get_dimensions(table['props'])
#         print(row_length, col_length)
#         print(dimensions)
                    
    string_matched_data[index]=table

In [10]:
## come here for only ds distant supervision

In [11]:
ds_train_data = pickle.load(open(os.path.join(table_dir, 'all_train_data_pre_upd.pkl'), 'rb'))

In [12]:
count_col, count_row = 0,0
for index, table in enumerate(ds_train_data):
    if len(table['anno_type'])==1:
        table['anno_type'] = 'dist_supv'
    else:
        table['anno_type'] = 'dist_supv__string_match'
    orient_list = table['prop_orient']
    assert isinstance(orient_list, str)
    # Count occurrences of each string in the list
#     string_counts = Counter(orient_list)

#     # Find the string with the maximum count
#     max_occuring_string = max(string_counts, key=string_counts.get)
    if orient_list == 'col':
        count_col+= 1
    elif orient_list == 'row':
        count_row+=1
    
#     table['prop_orient'] = max_occuring_string
    ds_train_data[index] = table
    
print(count_col, count_row)

374 51


In [13]:
combined_train_data_pre = ds_train_data + string_matched_data
# combined_train_data_pre = string_matched_data

# combined_train_data_pre = ds_train_data
print(len(combined_train_data_pre))
#print(combined_train_data_pre[453]['prop_orient'])

1021


In [14]:
save_data = '../data/data_in_sep_pkl'
if os.path.exists(save_data):
    shutil.rmtree(save_data)
    print(f"Directory removed.")
os.mkdir(save_data)
print(f"Directory created.")

for table in combined_train_data_pre:
    pii = table['pii']
    tid = table['t_idx']
    save_dir = os.path.join(save_data, pii+'__'+str(tid))
    if not os.path.exists(save_dir):
        os.mkdir(save_dir)
#     pickle.dump(table, open(os.path.join(save_dir, "init.pkl"), 'wb'))
    pickle.dump(table, open(os.path.join(save_dir, "init.pkl"), 'wb'))

Directory created.


In [15]:
pickle.dump(combined_train_data_pre, open(os.path.join(table_dir, 'train_data_prop_ds_and_anno.pkl'), 'wb'))

In [16]:
# print(string_matched_data[4].keys())
# print(string_matched_data[4])

In [17]:
# a = {'popp':[]}
# a['popp'].append([[[] for j in range(5)] for i in range(3)])
# print(a['popp'])
# print(a['popp'][-1])
# print(a['popp'][-1][-1])
# print(a['popp'][-1][2][3])
# a['popp'][-1][2][3].append((666, '5', ''))
# print(a['popp'])
# print(a['popp'][-1])

In [18]:
# ab = [(1,2),(2,3),(3,4)]
# print(ab.index((3,4)))

In [19]:
# row_label = [0, 1, 0]
# col_label = [0, 5, 0, 19, 0]
# result_array = [[(row, col) for col in col_label] for row in row_label]
# result_array

In [20]:
combined_train_data_pre[0]

{'act_table': [['Sample code',
   'Batch composition (mol%)',
   'Batch composition (mol%)',
   'Batch composition (mol%)',
   'Batch composition (mol%)',
   'Mo/P',
   'Molar ratio',
   'Molar ratio',
   'Sr/P',
   'T g (+-5 K)',
   'T f (+-5 K)',
   'a    (+-3.0x10-7/K)',
   'D (gcm-3) (+-0.5%)'],
  ['Sample code',
   'MoO3',
   'Fe2O3',
   'SrO',
   'P2O5',
   '',
   'O/P',
   'Fe/P',
   '',
   'T g (+-5 K)',
   'T f (+-5 K)',
   'a    (+-3.0x10-7/K)',
   'D (gcm-3) (+-0.5%)'],
  ['F40',
   '-',
   '40',
   '-',
   '60',
   '-',
   '3.5',
   '0.67',
   '-',
   '763',
   '789',
   '63.9',
   '3.00'],
  ['', '', '', '', '', '', '', '', '', '', '', '', ''],
  ['Series A',
   'Series A',
   'Series A',
   'Series A',
   'Series A',
   'Series A',
   'Series A',
   'Series A',
   'Series A',
   'Series A',
   'Series A',
   'Series A',
   'Series A'],
  ['A1',
   '5',
   '38',
   '-',
   '57',
   '0.04',
   '3.6',
   '0.67',
   '-',
   '762',
   '800',
   '67.7',
   '2.89'],
  ['A2',
   

In [21]:
combined_train_data_pre[0].keys()

dict_keys(['act_table', 'caption', 'row_label', 'col_label', 'edge_list', 'pii', 't_idx', 'regex_table', 'num_rows', 'num_cols', 'num_cells', 'comp_table', 'input_ids', 'attention_mask', 'caption_input_ids', 'caption_attention_mask', 'sum_less_100', 'footer', 'gid_row_label', 'gid_col_label', 'table_nums', 'prop_table', 'prop_names', 'prop_orient', 'props', 'prop_row_col_indexes', 'glass_id_type', 'anno_type'])

In [23]:
cd, ca, cda = 0, 0, 0
for w_table in combined_train_data_pre:
    if w_table['anno_type'] == 'dist_supv':
        cd+=1
    elif w_table['anno_type'] == 'string_match':
        ca+=1
    elif w_table['anno_type'] == 'dist_supv__string_match':
        cda+=1
        
print(cd, ca, cda)

330 596 95


In [26]:
act_energy = []
melt_temp = []
for w_table in combined_train_data_pre:
    id_key = w_table['pii'] + '_' + str(w_table['t_idx'])
#     if w_table['anno_type'] != 'dist_supv':
    if w_table['anno_type'] == 'string_match':
        if 23 in w_table['row_label'] or 23 in w_table['col_label']:
            act_energy.append(id_key)
        elif 14 in w_table['row_label'] or 14 in w_table['col_label']:
            melt_temp.append(id_key)
            
print(len(act_energy), len(melt_temp))

68 85


In [28]:
pickle.dump(act_energy, open('activation_energy_list.pkl', 'wb'))
pickle.dump(melt_temp, open('melting_temp_list.pkl', 'wb'))

In [31]:
my_upd_dict = {prop_names[i]: 0 for i in range(20)}
all_rows, all_cols = [], []
for table in combined_train_data_pre:
    all_rows += table['row_label']
    all_cols += table['col_label']
print('No of rows detected for each prop:')
for keys in range(4, 24):
    my_upd_dict[prop_names[keys-4]] = all_rows.count(keys) + all_cols.count(keys)
print(my_upd_dict)

No of rows detected for each prop:
{'Density': 359, 'GlassTransitionTg': 497, 'RefractiveIndex': 174, 'AbbeValue': 11, 'YoungsModulus': 74, 'ShearModulus': 17, 'VickersHardness': 55, 'PoissonRatio': 29, 'FractureToughness': 14, 'CrystallizationTemp': 418, 'MeltingTemp': 118, 'ElectricConduct': 52, 'DielectricConst': 13, 'TSofP': 16, 'TAnnealingP': 32, 'ExpansionCoeff': 80, 'LiquidusTemperature': 76, 'SofP': 18, 'BulkModulus': 21, 'ActivationEnergy': 119}


In [33]:
'''
For DS:
No of rows detected for each prop:
{'Density': 174, 'GlassTransitionTg': 252, 'RefractiveIndex': 92, 'AbbeValue': 3, 'YoungsModulus': 19, 'ShearModulus': 6, 'VickersHardness': 8, 'PoissonRatio': 13, 'FractureToughness': 6, 'CrystallizationTemp': 141, 'MeltingTemp': 0, 'ElectricConduct': 21, 'DielectricConst': 5, 'TSofP': 9, 'TAnnealingP': 5, 'ExpansionCoeff': 19, 'LiquidusTemperature': 31, 'SofP': 9, 'BulkModulus': 6, 'ActivationEnergy': 0}

For total:
{'Density': 359, 'GlassTransitionTg': 497, 'RefractiveIndex': 174, 'AbbeValue': 11, 'YoungsModulus': 74, 'ShearModulus': 17, 'VickersHardness': 55, 'PoissonRatio': 29, 'FractureToughness': 14, 'CrystallizationTemp': 418, 'MeltingTemp': 118, 'ElectricConduct': 52, 'DielectricConst': 13, 'TSofP': 16, 'TAnnealingP': 32, 'ExpansionCoeff': 80, 'LiquidusTemperature': 76, 'SofP': 18, 'BulkModulus': 21, 'ActivationEnergy': 119}

'''

"\nFor DS:\nNo of rows detected for each prop:\n{'Density': 174, 'GlassTransitionTg': 252, 'RefractiveIndex': 92, 'AbbeValue': 3, 'YoungsModulus': 19, 'ShearModulus': 6, 'VickersHardness': 8, 'PoissonRatio': 13, 'FractureToughness': 6, 'CrystallizationTemp': 141, 'MeltingTemp': 0, 'ElectricConduct': 21, 'DielectricConst': 5, 'TSofP': 9, 'TAnnealingP': 5, 'ExpansionCoeff': 19, 'LiquidusTemperature': 31, 'SofP': 9, 'BulkModulus': 6, 'ActivationEnergy': 0}\n\nFor total:\n{'Density': 359, 'GlassTransitionTg': 497, 'RefractiveIndex': 174, 'AbbeValue': 11, 'YoungsModulus': 74, 'ShearModulus': 17, 'VickersHardness': 55, 'PoissonRatio': 29, 'FractureToughness': 14, 'CrystallizationTemp': 418, 'MeltingTemp': 118, 'ElectricConduct': 52, 'DielectricConst': 13, 'TSofP': 16, 'TAnnealingP': 32, 'ExpansionCoeff': 80, 'LiquidusTemperature': 76, 'SofP': 18, 'BulkModulus': 21, 'ActivationEnergy': 119}\n\n"